# Can we predict late delivery?
Target: `late_delivery`. Follows `checklist.md`.

Decisions up front:
- **Prediction time = dispatch.** `claim_paid_eur` is paid on a claim settled *after* delivery (mean about 78 EUR for late vs about 1 EUR for on-time), so it leaks the outcome and is dropped. `consignment` is an identifier and is dropped.
- **Split by time.** A model would be used on later consignments, so the test set is the latest 20% of rows. Each consignment id appears once, so no entity grouping is needed.
- **CV** inside the training rows uses expanding-window forward splits (earlier rows train, later rows validate).

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.base import clone

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
assert df.consignment.is_unique
df['hour'] = df.dispatched_on.dt.hour
df['weekday'] = df.dispatched_on.dt.dayofweek
print(df.shape, df.isna().sum().sum(), 'missing')

## 1. Set aside the test set (latest 20% of rows)

In [ ]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.dispatched_on.min(), train.dispatched_on.max(), len(train))
print(test.dispatched_on.min(), test.dispatched_on.max(), len(test))
# Exploration uses training rows only
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3).to_string())

The late rate drifts a lot from month to month, so calendar month is **not** used as a feature; the model has to rely on shipment attributes. Expect modest performance under time shift.

In [ ]:
FEATURES = ['carrier', 'distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']  # all known at dispatch
X, y = train[FEATURES], train.late_delivery
Xt, yt = test[FEATURES], test.late_delivery
print('train late rate', y.mean().round(3))

## 4. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: logistic regression, one-hot carrier, scaled numerics; C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting; (max_depth, learning_rate) in {(2, 0.05), (3, 0.05), (3, 0.1)}, 150 iterations.

Selection metric: mean forward-CV average precision. Threshold: the one maximising F1 on pooled out-of-fold predictions of the chosen model. Preprocessing sits inside each Pipeline, so it is fit per fold (checklist 3).

In [ ]:
num = ['distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']
def prep(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
cands = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for c_ in [0.1, 1, 10]:
    cands[f'logreg_C{c_}'] = Pipeline([('p', prep(True)), ('m', LogisticRegression(C=c_, max_iter=1000))])
for d, lr in [(2, 0.05), (3, 0.05), (3, 0.1)]:
    cands[f'hgb_d{d}_lr{lr}'] = Pipeline([('p', prep(False)),
        ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=lr, max_iter=150, random_state=0))])
cv = TimeSeriesSplit(n_splits=5)
oof, rows = {}, []
for name, pipe in cands.items():
    p = np.full(len(train), np.nan); ap = []; auc = []
    for tr, va in cv.split(X):
        m = clone(pipe).fit(X.iloc[tr], y.iloc[tr])
        p[va] = m.predict_proba(X.iloc[va])[:, 1]
        ap.append(average_precision_score(y.iloc[va], p[va]))
        auc.append(roc_auc_score(y.iloc[va], p[va]))
    oof[name] = p; rows.append((name, np.mean(ap), np.std(ap), np.mean(auc)))
res = pd.DataFrame(rows, columns=['model', 'cv_AP', 'cv_AP_sd', 'cv_AUC']).sort_values('cv_AP', ascending=False)
res.round(3)

## 5. Choose model and threshold on CV only

In [ ]:
best = res.iloc[0].model
mask = ~np.isnan(oof[best])
ths = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(y[mask], oof[best][mask] > t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr, 2), '| CV F1:', round(max(f1s), 3))

**Recorded choice (from the CV output above, before the test set is touched):** the model printed above, with the printed F1-optimal threshold. No tuning after this point.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = clone(cands[best]).fit(X, y)
pt = final.predict_proba(Xt)[:, 1]
pred = pt > thr
print('test late rate', yt.mean().round(3))
print(pd.Series({'AUC': roc_auc_score(yt, pt), 'AP': average_precision_score(yt, pt),
                 'baseline AP (= prevalence)': yt.mean(),
                 'precision': precision_score(yt, pred, zero_division=0),
                 'recall': recall_score(yt, pred), 'F1': f1_score(yt, pred)}).round(3))

## Interpretation
Compare the headline numbers with the baseline AP (the test prevalence). Caveat: the late rate swings strongly over time and falls to near zero in the final months. That may be real change or labels not yet settled for recent shipments. Either way, a threshold tuned on earlier months can be miscalibrated on the test period. The test set was scored once.